In this notebook I test differential framework

In [3]:
# an example with nabla_at
from zhai2022.differential import nabla_at
import torch

# nabla_at is a simple differentiation
R = torch.rand(5,4)
def f(x: torch.tensor):
    return torch.einsum('ij,nj->ni',R,x)  # a linear function
x = torch.rand(4)  # a vector with shape (4)
df_dx = nabla_at(f,x)  # shape: (5,4)
print(f'Results: \n {df_dx}')
# in particular, df_i/dx_j = df_dx[i,j]
print(f'Expectation: \n {R}')

Results: 
 tensor([[0.5549, 0.6640, 0.1372, 0.3305],
        [0.7873, 0.2060, 0.6352, 0.2688],
        [0.8653, 0.2346, 0.7773, 0.6412],
        [0.4946, 0.7647, 0.3001, 0.1818],
        [0.4297, 0.3228, 0.4888, 0.6374]])
Expectation: 
 tensor([[0.5549, 0.6640, 0.1372, 0.3305],
        [0.7873, 0.2060, 0.6352, 0.2688],
        [0.8653, 0.2346, 0.7773, 0.6412],
        [0.4946, 0.7647, 0.3001, 0.1818],
        [0.4297, 0.3228, 0.4888, 0.6374]])


In [4]:
# an example with div_at

from zhai2022.differential import div_at
import torch

# nabla_at is a simple differentiation
R = torch.rand(5,4,4)
def f(x: torch.tensor):
    return torch.einsum('kij,nj->nki',R,x)  # a linear function
x = torch.rand(4)  # a vector with shape (4)
divf = div_at(f,x)  # shape: (5,4)
print(f'Results: \n {divf}')
# in particular, df_i/dx_j = df_dx[i,j]
print(f'Expectation: \n {torch.einsum("iaa->i", R)}')

Results: 
 tensor([1.4562, 2.2161, 1.8308, 2.5274, 2.8251])
Expectation: 
 tensor([1.4562, 2.2161, 1.8308, 2.5274, 2.8251])


**Differential** is a class used for differential oeprations.

Here, some example:

In [10]:
# an example with Differential.Nabla

from zhai2022.differential import Differential
import torch

dif = Differential(2)
# dif is formally a nabla operator with order 2
# the common usage is to compute second order derivatives
s = torch.rand(4,4)
S2 = s.T @ s
v = torch.rand(4)
def f(x: torch.tensor):
    return torch.einsum('ni,ij,nj->n',x,S2,x)/2 + torch.einsum('i,ni->n',v,x)  # a quadratic form

x = torch.rand(2,4)  # a vector with shape (2,4) -> a batch of 2 vectors
d2f_dx2 = dif(f)(x)
print(f'Expectation: \n {S2} (repeated 2 times along a new leading dimension)')
print(f'Results: \n {d2f_dx2}')

# if you want, you can also compute all derivatives up to order 2
res = dif(f)(x, full_trace=True)  # returns a tuple
f_x = res[0]  # shape: (2,)
df_dx = res[1]  # shape: (2,4)
d2f_dx2_full = res[2]  # shape: (2,4,4)
print(f'Function values: \n {f_x}')
print(f'First derivatives: \n {df_dx}')
print(f'Second derivatives (full): \n {d2f_dx2_full}')

Expectation: 
 tensor([[0.7330, 1.1384, 0.8942, 0.4798],
        [1.1384, 1.9587, 1.5954, 0.6723],
        [0.8942, 1.5954, 1.5639, 0.1832],
        [0.4798, 0.6723, 0.1832, 1.0083]]) (repeated 2 times along a new leading dimension)
Results: 
 (tensor([[[0.7330, 1.1384, 0.8942, 0.4798],
         [1.1384, 1.9587, 1.5954, 0.6723],
         [0.8942, 1.5954, 1.5639, 0.1832],
         [0.4798, 0.6723, 0.1832, 1.0083]],

        [[0.7330, 1.1384, 0.8942, 0.4798],
         [1.1384, 1.9587, 1.5954, 0.6723],
         [0.8942, 1.5954, 1.5639, 0.1832],
         [0.4798, 0.6723, 0.1832, 1.0083]]]),)
Function values: 
 tensor([2.4401, 4.0407])
First derivatives: 
 tensor([[1.7288, 2.9668, 1.9512, 1.7173],
        [2.3004, 3.8341, 2.6034, 2.1629]], grad_fn=<StackBackward0>)
Second derivatives (full): 
 tensor([[[0.7330, 1.1384, 0.8942, 0.4798],
         [1.1384, 1.9587, 1.5954, 0.6723],
         [0.8942, 1.5954, 1.5639, 0.1832],
         [0.4798, 0.6723, 0.1832, 1.0083]],

        [[0.7330, 1.1384, 

In [1]:
# an example with Differential.Div

from zhai2022.differential import Differential
import torch

dif = Differential(2)
# dif is formally a nabla operator with order 2
# the common usage is to compute second order derivatives
s = torch.rand(4,4)
# f: R^{n x 2} -> R^{n x 2 x 2}
def f(x: torch.tensor):
    return torch.stack(
        (
            torch.stack((x[:,0]**2+x[:,1]**2, x[:,0]+x[:,1]), dim=-1),  # row 0
            torch.stack((x[:,0]*x[:,1], x[:,0]-x[:,1]), dim=-1)         # row 1
        ), dim=-2
    )

x = torch.rand(1,2)  # a vector with shape (3,2) -> a batch of 3 vectors
div2f = dif.div(f)   # this is a callable function that computes the divergence of order 2
res = div2f(x, full_trace=True)  # returns a tuple
f_x = res[0]  # shape: (n,2,2)
divf_x = res[1]  # shape: (n,2)
div2f_x = res[2]  # shape: (n,)

# expectation:
print('Expectation:')
# Function values:
# x0**2 + x1**2  x0 + x1
#     x0*x1      x0 - x1
x0 = x[0,0].item()
x1 = x[0,1].item()
print('# Function values:')
print(f'# {x0**2 + x1**2 = }  {x0 + x1 = }')
print(f'#         {x0*x1 = }  {x0 - x1 = }')
print(f'Function values: \n {f_x}')
# First divergence:
# 2*x0 + 1
# x1 - 1
print('# First divergence:')
print(f'# {2*x0 + 1 = }')
print(f'#   {x1 - 1 = }')
print(f'First divergence: \n {divf_x}')
# Second divergence (full):
# 2 + 1 = 3
print('# Second divergence (full):')
print(f'# {2 + 1 = }')
print(f'Second divergence (full): \n {div2f_x}')

Expectation:
# Function values:
# x0**2 + x1**2 = 0.38624325950607385  x0 + x1 = 0.7240562438964844
#         x0*x1 = 0.0690070924097057  x0 - x1 = -0.4982259273529053
Function values: 
 tensor([[[ 0.3862,  0.7241],
         [ 0.0690, -0.4982]]])
# First divergence:
# 2*x0 + 1 = 1.225830316543579
#   x1 - 1 = -0.3888589143753052
First divergence: 
 tensor([[ 1.2258, -0.3889]], grad_fn=<SqueezeBackward1>)
# Second divergence (full):
# 2 + 1 = 3
Second divergence (full): 
 3.0
